In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path("../src").resolve()))

from config import (
    RAW_ROOT,
    GROUP1_PATIENTS,
    GROUP2_PATIENTS,
    GROUP3_PATIENTS,
    BAD_FILES,
    CHANNEL_ORDER
)

from utils import load_raw_edf, prepare_raw

def get_group_valid_edf_files(patient_list, bad_files_dict=None):
    valid_files = []

    for patient_name in patient_list:
        patient_folder = RAW_ROOT / patient_name
        edf_files = sorted(patient_folder.glob("*.edf"))

        patient_bad_files = set()
        if bad_files_dict is not None:
            patient_bad_files = set(bad_files_dict.get(patient_name, []))

        for file_path in edf_files:
            if file_path.name in patient_bad_files:
                continue
            valid_files.append(file_path)

    return valid_files

def find_common_channels_for_group(valid_files, channel_order):
    if len(valid_files) == 0:
        return [], set()

    first_raw = load_raw_edf(valid_files[0])
    first_raw = prepare_raw(first_raw)
    common_set = set(first_raw.ch_names)
    reference_channels = first_raw.ch_names.copy()

    for file_path in valid_files[1:]:
        raw = load_raw_edf(file_path)
        raw = prepare_raw(raw)
        current_set = set(raw.ch_names)
        common_set = common_set.intersection(current_set)

    ordered_common = [ch for ch in channel_order if ch in common_set]

    extra_channels = [
        ch for ch in reference_channels
        if ch in common_set and ch not in ordered_common
    ]
    ordered_common.extend(extra_channels)

    return ordered_common, common_set


In [4]:
group1_valid_files = get_group_valid_edf_files(GROUP1_PATIENTS)
group2_valid_files = get_group_valid_edf_files(GROUP2_PATIENTS)
group3_valid_files = get_group_valid_edf_files(GROUP3_PATIENTS, BAD_FILES)

group1_common_ordered, group1_common_set = find_common_channels_for_group(group1_valid_files, CHANNEL_ORDER)
group2_common_ordered, group2_common_set = find_common_channels_for_group(group2_valid_files, CHANNEL_ORDER)
group3_common_ordered, group3_common_set = find_common_channels_for_group(group3_valid_files, CHANNEL_ORDER)

print("Group 1 common channels:")
print(f"Count: {len(group1_common_ordered)}")
print(group1_common_ordered)

print("\nGroup 2 common channels:")
print(f"Count: {len(group2_common_ordered)}")
print(group2_common_ordered)

print("\nGroup 3 common channels:")
print(f"Count: {len(group3_common_ordered)}")
print(group3_common_ordered)

C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
C:\Users\MSI\Desktop\EEG_FYP\src\utils.py:20: RuntimeWarning: Ch

Group 1 common channels:
Count: 23
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'T8-P8-0', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']

Group 2 common channels:
Count: 23
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'P8-O2', 'FZ-CZ', 'CZ-PZ', 'P7-T7', 'T7-FT9', 'T8-P8-0', 'FT9-FT10', 'FT10-T8', 'T8-P8-1']

Group 3 common channels:
Count: 21
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'P8-O2', 'FZ-CZ', 'CZ-PZ', '--0', '--1', '--2', '--3']


In [5]:
group123_shared = group1_common_set.intersection(group2_common_set).intersection(group3_common_set)
group123_shared_ordered = [ch for ch in CHANNEL_ORDER if ch in group123_shared]

print("Common channels shared across Group 1, Group 2, and Group 3:")
print(f"Count: {len(group123_shared_ordered)}")
print(group123_shared_ordered)

Common channels shared across Group 1, Group 2, and Group 3:
Count: 17
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']


In [6]:
print("Channels in Group 1 but not in Group 2:")
print(sorted(group1_common_set - group2_common_set))

print("\nChannels in Group 1 but not in Group 3:")
print(sorted(group1_common_set - group3_common_set))

print("\nChannels in Group 2 but not in Group 3:")
print(sorted(group2_common_set - group3_common_set))

Channels in Group 1 but not in Group 2:
[]

Channels in Group 1 but not in Group 3:
['FT10-T8', 'FT9-FT10', 'P7-T7', 'T7-FT9', 'T8-P8-0', 'T8-P8-1']

Channels in Group 2 but not in Group 3:
['FT10-T8', 'FT9-FT10', 'P7-T7', 'T7-FT9', 'T8-P8-0', 'T8-P8-1']


In [7]:
from pathlib import Path
import pandas as pd
import sys
import mne

sys.path.append(str(Path("../src").resolve()))

from config import (
    RAW_ROOT,
    GROUP1_PATIENTS,
    GROUP2_PATIENTS,
    GROUP3_PATIENTS,
    GROUP4_PATIENTS,
    BAD_FILES
)

final_21_channels =['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 
 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 
 'FP2-F8', 'F8-T8', 'P8-O2', ''
 'FZ-CZ', 'CZ-PZ']

VALID_BIPOLAR_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ", "P7-T7", "T7-FT9",
    "FT9-FT10", "FT10-T8"
]
print("Final 21-channel set:")
print(final_21_channels)
print("Count:", len(final_21_channels))

#funcation that check for the 21 channel plan 

def get_eeg_channels(raw):
    valid_set = set(VALID_BIPOLAR_CHANNELS)
    eeg_channels = [ch for ch in raw.ch_names if ch in valid_set]
    return eeg_channels

all_patients = (
    GROUP1_PATIENTS +
    GROUP2_PATIENTS +
    GROUP3_PATIENTS +
    GROUP4_PATIENTS
)

rows = []

for patient_name in all_patients:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))
    patient_bad_files = set(BAD_FILES.get(patient_name, []))

    valid_files = [f for f in edf_files if f.name not in patient_bad_files]

    if len(valid_files) == 0:
        rows.append({
            "patient_name": patient_name,
            "n_valid_files": 0,
            "supports_full_21": False,
            "n_overlap_with_21": 0,
            "missing_channels": "NO VALID FILES"
        })
        continue

    patient_common = None

    for file_path in valid_files:
        ch_names = set(get_eeg_channels_fast(file_path))

        if patient_common is None:
            patient_common = ch_names
        else:
            patient_common = patient_common.intersection(ch_names)

        # early stop if already impossible to reach full 21
        if len(patient_common.intersection(final_21_set)) < len(final_21_set):
            # keep going only if you want exact final common set
            # here we continue because we still want exact missing list
            pass

    overlap = patient_common.intersection(final_21_set)
    missing = final_21_set - patient_common

    rows.append({
        "patient_name": patient_name,
        "n_valid_files": len(valid_files),
        "n_patient_common_channels": len(patient_common),
        "n_overlap_with_21": len(overlap),
        "supports_full_21": len(overlap) == len(final_21_set),
        "missing_channels": ", ".join([ch for ch in final_21_channels if ch in missing])
    })

df_21_support = pd.DataFrame(rows)
display(df_21_support.sort_values(["supports_full_21", "n_overlap_with_21"], ascending=[False, False]))

print("Patients that CAN support the full 21-channel plan:")
display(df_21_support[df_21_support["supports_full_21"] == True])

print("\nPatients that CANNOT support the full 21-channel plan:")
display(df_21_support[df_21_support["supports_full_21"] == False])

Final 21-channel set:
['FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'FP2-F8', 'F8-T8', 'P8-O2', 'FZ-CZ', 'CZ-PZ']
Count: 17


C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2062943413.py:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2062943413.py:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2062943413.py:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2062943413.py:35: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=Fa

,patient_name,n_valid_files,n_patient_common_channels,n_overlap_with_21,supports_full_21,missing_channels
0,chb01,42,23,20,False,
1,chb02,36,23,20,False,
2,chb03,38,23,20,False,
3,chb04,42,23,20,False,
4,chb05,39,23,20,False,
5,chb06,18,23,20,False,
6,chb07,19,23,20,False,
7,chb08,20,23,20,False,
8,chb09,19,23,20,False,
9,chb10,25,23,20,False,


Patients that CAN support the full 21-channel plan:


,patient_name,n_valid_files,n_patient_common_channels,n_overlap_with_21,supports_full_21,missing_channels



Patients that CANNOT support the full 21-channel plan:


,patient_name,n_valid_files,n_patient_common_channels,n_overlap_with_21,supports_full_21,missing_channels
0,chb01,42,23,20,False,
1,chb02,36,23,20,False,
2,chb03,38,23,20,False,
3,chb04,42,23,20,False,
4,chb05,39,23,20,False,
5,chb06,18,23,20,False,
6,chb07,19,23,20,False,
7,chb08,20,23,20,False,
8,chb09,19,23,20,False,
9,chb10,25,23,20,False,


From the analysis shown, it is clear that using a fixed 21-channel standard across all patients is not fully suitable. Although some patients (especially from Group 3 like chb16 and chb17) match the 21-channel structure, many other patients do not consistently support it. For example, Group 1 and Group 2 patients mostly have 23 or even 28 channels, which means they contain extra channels that would be lost if forced into 21 channels. On the other hand, some Group 4 patients (such as chb18 and chb12) have fewer or inconsistent channels, with chb12 showing very low overlap and chb18 only supporting 17 channels. This inconsistency means that enforcing a 21-channel setup would either require removing useful information from richer datasets or excluding patients with missing channels. Therefore, the 21-channel standard does not fit well across all groups because of variation in channel counts, missing channels, and differences in recording setups between patients, making it difficult to create a unified dataset without losing data or patients.

so the choice is making a 17 channels set 
first we should define a final 17 channels then verifiy including patients support them then using a patients split - load the processed files only from train /val/test patients 
reduce to the 17 channel oreder then combine 
- X_train, y_train
- X_val, y_val
- X_test, y_test 

then balance only for the training ( using undersample and applying lass weihhts)
train model + evaluate on vaildation and testing data.

In [8]:
# verifiying the 17 channel set
from pathlib import Path
import pandas as pd
import sys
import mne

sys.path.append(str(Path("../src").resolve()))

from config import (
    RAW_ROOT,
    GROUP1_PATIENTS,
    GROUP2_PATIENTS,
    GROUP3_PATIENTS,
    GROUP4_PATIENTS,
    BAD_FILES
)

FINAL_17_CHANNELS = [
    "FP1-F7", "F7-T7", "T7-P7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "P8-O2",
    "FZ-CZ", "CZ-PZ"
]

FINAL_17_SET = set(FINAL_17_CHANNELS)

def get_eeg_channels_fast(file_path):
    raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
    valid_channels = [ch for ch in raw.ch_names if ch in FINAL_17_SET or "-" in ch]
    return raw.ch_names

all_patients = GROUP1_PATIENTS + GROUP2_PATIENTS + GROUP3_PATIENTS + GROUP4_PATIENTS

rows = []

for patient_name in all_patients:
    patient_folder = RAW_ROOT / patient_name
    edf_files = sorted(patient_folder.glob("*.edf"))
    patient_bad_files = set(BAD_FILES.get(patient_name, []))
    valid_files = [f for f in edf_files if f.name not in patient_bad_files]

    if len(valid_files) == 0:
        rows.append({
            "patient_name": patient_name,
            "n_valid_files": 0,
            "supports_final_17": False,
            "missing_channels": "NO VALID FILES"
        })
        continue

    patient_common = None

    for file_path in valid_files:
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
        current_channels = set(raw.ch_names)

        if patient_common is None:
            patient_common = current_channels
        else:
            patient_common = patient_common.intersection(current_channels)

    overlap = patient_common.intersection(FINAL_17_SET)
    missing = FINAL_17_SET - patient_common

    rows.append({
        "patient_name": patient_name,
        "n_valid_files": len(valid_files),
        "n_overlap_with_final_17": len(overlap),
        "supports_final_17": len(overlap) == len(FINAL_17_SET),
        "missing_channels": ", ".join([ch for ch in FINAL_17_CHANNELS if ch in missing])
    })

df_final17_check = pd.DataFrame(rows)
display(df_final17_check.sort_values(["supports_final_17", "n_overlap_with_final_17"], ascending=[False, False]))

C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2375790570.py:55: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2375790570.py:55: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2375790570.py:55: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\MSI\AppData\Local\Temp\ipykernel_13132\2375790570.py:55: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=Fa

,patient_name,n_valid_files,n_overlap_with_final_17,supports_final_17,missing_channels
0,chb01,42,17,True,
1,chb02,36,17,True,
2,chb03,38,17,True,
3,chb04,42,17,True,
4,chb05,39,17,True,
5,chb06,18,17,True,
6,chb07,19,17,True,
7,chb08,20,17,True,
8,chb09,19,17,True,
9,chb10,25,17,True,


In [1]:
import numpy as np
from pathlib import Path

npz_path = Path(r"D:\EEG_FYP_DATA\processed\group1\chb01\chb01_03_processed.npz")

data = np.load(npz_path)
X = data["X"]
y = data["y"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))
print("Label counts:")

unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"Label {label}: {count}")

X shape: (1439, 23, 1280)
y shape: (1439,)
Unique labels: [0 1]
Label counts:
Label 0: 1421
Label 1: 18


IndentationError: unexpected indent (2443149599.py, line 28)